# Der ganze Ablauf: Train, Validation, Test und Cross-Validation

Dieses Notebook spielt den kompletten Ablauf mit **einem** Datensatz durch, Schritt für Schritt und mit sichtbaren Zwischenergebnissen.

| Teil | Inhalt |
|---|---|
| **A** | Einfacher Split 60 / 20 / 20, jeder Schritt von Hand |
| **B** | Dieselbe Aufgabe mit Cross-Validation (CV), Durchlauf für Durchlauf |
| **C** | CV zum Vergleichen von Vorverarbeitungen, inklusive falschem Vorgehen (Leakage) |
| **D** | CV bei Klassifikation: `StratifiedKFold` |
| **E** | CV bei Zeitreihen: `TimeSeriesSplit` |

Die Regeln, die überall gelten:

1. **Zuerst** das Testset abtrennen. Es wird nur **einmal ganz am Schluss** benutzt.
2. Imputer und Scaler werden **nur auf dem Trainingsteil** angepasst (`fit`). Auf Validation und Test gibt es nur `transform`.
3. Reihenfolge der Vorverarbeitung: erst fehlende Werte auffüllen, dann skalieren.
4. Auswahl (Hyperparameter, Strategie) geschieht auf der Validation bzw. in der CV.
5. Danach wird das gewählte **Setup** einmal auf Training + Validation neu trainiert und auf dem Test bewertet.

Benötigt: `numpy`, `pandas`, `scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42  # every random choice below uses this seed

## Die Daten

25 Wohnungen mit drei Merkmalen und dem Preis (in 1000 CHF) als Ziel. In einigen Zellen fehlen Werte (`NaN`). Die Daten sind erfunden, aber so gebaut, dass der Preis wirklich von den Merkmalen abhängt.

In [ ]:
rng = np.random.default_rng(SEED)
n = 25
rooms = rng.integers(2, 8, n)
area = (rooms * 18 + rng.normal(0, 8, n)).round()
age = rng.integers(1, 60, n)
price = (100 + 3 * area - 2 * age + rng.normal(0, 15, n)).round()

data = pd.DataFrame({"Zimmer": rooms, "Flaeche": area, "Alter": age}, index=pd.RangeIndex(1, n + 1, name="ID")).astype(float)
for row_id, column in [(3, "Flaeche"), (11, "Flaeche"), (19, "Flaeche"), (7, "Alter"), (14, "Alter"),
                       (22, "Alter"), (5, "Zimmer"), (16, "Zimmer"), (24, "Alter")]:
    data.loc[row_id, column] = np.nan
y = pd.Series(price, index=data.index, name="Preis")

print("Fehlende Werte pro Spalte:", data.isna().sum().to_dict())
data.join(y).head(10)

# Teil A: Einfacher Split (60 / 20 / 20)

## A1 · Aufteilen

Zuerst werden 20 % als **Test** abgetrennt. Die übrigen 80 % (die *Entwicklungsdaten*) werden im Verhältnis 75 : 25 in **Training** und **Validation** geteilt. Das ergibt 60 % / 20 % / 20 % der Gesamtdaten, also 15 / 5 / 5 Zeilen.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(data, y, test_size=0.20, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, random_state=SEED)

part = pd.Series("Training", index=data.index, name="Teil")
part.loc[X_val.index] = "Validation"
part.loc[X_test.index] = "Test"
pd.DataFrame({
    "Anzahl": part.value_counts(),
    "IDs": part.groupby(part).apply(lambda ids: sorted(ids.index)),
    "Zeilen mit fehlenden Werten": part[data.isna().any(axis=1)].groupby(part).apply(lambda ids: sorted(ids.index)),
}).loc[["Training", "Validation", "Test"]]

Lesehilfe: In jedem Teil kommen fehlende Werte vor. Das Testset wird nicht ausgespart: Auch dort muss später aufgefüllt werden, aber mit Werten aus dem Training.

## A2 · Fehlende Werte auffüllen

Strategie festlegen (`median`) und den Imputer **nur auf dem Training** anpassen. `fit` lernt pro Spalte den Median der vorhandenen Trainingswerte.

In [ ]:
imputer = SimpleImputer(strategy="median").set_output(transform="pandas")
imputer.fit(X_train)

train_medians = pd.Series(imputer.statistics_, index=X_train.columns, name="Trainingsmedian")
train_medians

Jetzt `transform` auf alle drei Teile. Die Tabelle zeigt jede Zelle, die aufgefüllt wurde, und den Ersatzwert. Validation (ID 14) und Test (ID 24) erhalten dieselben Trainingsmediane, hier `Alter = 26`.

In [ ]:
def filled_cells(raw, clean, name):
    # list every cell that was NaN before, with its replacement value
    rows = [(name, row_id, column, clean.loc[row_id, column])
            for row_id in raw.index for column in raw.columns if pd.isna(raw.loc[row_id, column])]
    return pd.DataFrame(rows, columns=["Teil", "ID", "Spalte", "Ersatzwert"])


X_train_imp = imputer.transform(X_train)
X_val_imp = imputer.transform(X_val)
X_test_imp = imputer.transform(X_test)

pd.concat([
    filled_cells(X_train, X_train_imp, "Training"),
    filled_cells(X_val, X_val_imp, "Validation"),
    filled_cells(X_test, X_test_imp, "Test"),
], ignore_index=True)

## A3 · Skalieren

Der `StandardScaler` zieht den Mittelwert ab und teilt durch die Standardabweichung. Auch er lernt diese zwei Zahlen **nur aus dem Training** (nach der Imputation).

In [ ]:
scaler = StandardScaler().set_output(transform="pandas")
scaler.fit(X_train_imp)

X_train_s = scaler.transform(X_train_imp)
X_val_s = scaler.transform(X_val_imp)
X_test_s = scaler.transform(X_test_imp)

pd.DataFrame({
    "Mittelwert (Training)": scaler.mean_,
    "Std (Training)": scaler.scale_,
    "Mittel nach Skalierung: Training": X_train_s.mean(),
    "Mittel nach Skalierung: Validation": X_val_s.mean(),
}, index=X_train.columns).round(2)

Lesehilfe: Im Training ist der skalierte Mittelwert exakt 0. In der Validation ist er **nicht** 0, weil sie mit den Trainingszahlen skaliert wird. Das ist richtig so. Würde man Validation separat skalieren, wären ihre Werte nicht mehr mit dem Training vergleichbar.

## A4 · Modelle trainieren und auf der Validation vergleichen

Wir probieren vier Werte für den Hyperparameter `alpha` (Stärke der Regularisierung bei Ridge). Jedes Modell lernt auf dem Training und wird auf der **Validation** bewertet (Fehler MAE, kleiner ist besser).

In [ ]:
alphas = [0.01, 1, 10, 100]
val_errors = {}
for alpha in alphas:
    model = Ridge(alpha=alpha).fit(X_train_s, y_train)
    val_errors[alpha] = mean_absolute_error(y_val, model.predict(X_val_s))

val_table = pd.Series(val_errors, name="Validation-MAE").rename_axis("alpha").round(1)
best_alpha = val_table.idxmin()
print("Bestes alpha laut Validation:", best_alpha)
val_table

Damit ist die Auswahl getroffen: Setup = Median-Imputation + Standardisierung + Ridge mit `best_alpha`. Der Test wurde bis hierher **nicht** angesehen.

## A5 · Finales Modell auf Training + Validation, dann Test

Das gewählte Setup wird auf allen Entwicklungsdaten neu angepasst: neuer Imputer, neuer Scaler, neues Modell. Danach gibt es **einen einzigen** Blick auf den Test.

In [ ]:
final_imputer = SimpleImputer(strategy="median").set_output(transform="pandas").fit(X_dev)
final_scaler = StandardScaler().set_output(transform="pandas").fit(final_imputer.transform(X_dev))
final_model = Ridge(alpha=best_alpha).fit(final_scaler.transform(final_imputer.transform(X_dev)), y_dev)

X_test_final = final_scaler.transform(final_imputer.transform(X_test))
test_mae = mean_absolute_error(y_test, final_model.predict(X_test_final))

print(f"Testfehler (MAE): {test_mae:.1f}")
pd.DataFrame({
    "Median: nur Training": train_medians,
    "Median: Training + Validation": pd.Series(final_imputer.statistics_, index=X_dev.columns),
})

Lesehilfe: Die Mediane (und ebenso Mittelwert und Std des Scalers sowie die Koeffizienten) sind im finalen Modell andere als in A2 bis A4. Das ist gewollt: Es ist ein neues Modell mit mehr Daten. Gleich geblieben sind nur die **Entscheidungen** (Median-Strategie, `best_alpha`). Der Testfehler gehört zu diesem finalen Modell.

## A6 · Kontrolle mit `Pipeline`

Eine `Pipeline` fasst Imputer, Scaler und Modell zusammen. `fit` passt alle Schritte der Reihe nach nur an den übergebenen Daten an, `predict` wendet sie an. Das Ergebnis muss mit A5 übereinstimmen.

In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=best_alpha)),
])
pipeline.fit(X_dev, y_dev)
pipeline_mae = mean_absolute_error(y_test, pipeline.predict(X_test))

assert abs(pipeline_mae - test_mae) < 1e-9
print(f"Pipeline-Testfehler: {pipeline_mae:.1f} (identisch zu A5)")

# Teil B: Dieselbe Aufgabe mit Cross-Validation

Statt **einer** Validation (nur 5 Zeilen, also viel Zufall) werden die 20 Entwicklungszeilen in **k = 5 Folds** zu je 4 Zeilen geteilt. Bei 5 Folds ist die Validation je 20 % der Entwicklungsdaten. Das Testset (dieselben 5 Zeilen wie in Teil A) bleibt komplett draussen.

## B1 · Wer ist wann im Training?

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(X_dev))  # positions inside X_dev

plan = pd.DataFrame("Train", index=data.index, columns=[f"Durchlauf {i}" for i in range(1, 6)])
plan.loc[X_test.index, :] = "Test"
for i, (train_pos, val_pos) in enumerate(folds, start=1):
    plan.loc[X_dev.index[val_pos], f"Durchlauf {i}"] = "Val"

assert ((plan.loc[X_dev.index] == "Val").sum(axis=1) == 1).all()  # each dev row is Val exactly once
plan

Lesehilfe: Lies jede Spalte als einen Durchlauf. In jedem Durchlauf sind 16 Zeilen `Train` und 4 Zeilen `Val`. In jeder Zeile (ID) steht genau einmal `Val`, also ist jede Entwicklungszeile in genau einem Durchlauf die Validierung. Die `Test`-Zeilen sind überall dieselben.

## B2 · Alle Kombinationen: alpha × Durchlauf

Für jeden Wert von `alpha` und jeden Durchlauf wird eine **frische** Pipeline trainiert (neuer Imputer, neuer Scaler, neues Modell), auf dem Validierungsfold bewertet und verworfen. Bei 4 Werten und 5 Folds sind das 20 Trainings.

In [ ]:
base_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge()),
])

cv_errors = pd.DataFrame(index=pd.Index(alphas, name="alpha"), columns=[f"Durchlauf {i}" for i in range(1, 6)], dtype=float)
for alpha in alphas:
    for i, (train_pos, val_pos) in enumerate(folds, start=1):
        fold_pipeline = clone(base_pipeline).set_params(model__alpha=alpha)
        fold_pipeline.fit(X_dev.iloc[train_pos], y_dev.iloc[train_pos])
        cv_errors.loc[alpha, f"Durchlauf {i}"] = mean_absolute_error(
            y_dev.iloc[val_pos], fold_pipeline.predict(X_dev.iloc[val_pos])
        )

cv_errors["Mittel"] = cv_errors.mean(axis=1)
cv_errors["Streuung"] = cv_errors.iloc[:, :5].std(axis=1, ddof=0)
best_alpha_cv = cv_errors["Mittel"].idxmin()
print("Bestes alpha laut CV:", best_alpha_cv)
cv_errors.round(1)

Lesehilfe: Jede Zeile ist ein Kandidat, jede Spalte ein Durchlauf. Der **Mittelwert** über die fünf Durchläufe entscheidet. Die Streuung zeigt, wie stabil das ist: Ein Kandidat, der in einem Fold sehr gut und in einem anderen sehr schlecht ist, ist weniger verlässlich als ein gleichmässiger.

## B3 · Finales Modell und Test

Wie in A5: Das beste Setup wird **einmal** auf allen 20 Entwicklungszeilen trainiert und einmal auf dem Test bewertet. Die fünf CV-Modelle sind verworfen.

In [ ]:
final_cv_model = clone(base_pipeline).set_params(model__alpha=best_alpha_cv).fit(X_dev, y_dev)
test_mae_cv = mean_absolute_error(y_test, final_cv_model.predict(X_test))
print(f"Erwartung laut CV (Mittel): {cv_errors.loc[best_alpha_cv, 'Mittel']:.1f}")
print(f"Testfehler (einmalig):      {test_mae_cv:.1f}")

Vergleich der drei Zahlen in diesem Beispiel: Die einzelne Validation aus Teil A (5 Zeilen) ergab für `alpha=10` einen Fehler von 9.7. Das CV-Mittel über alle 20 Entwicklungszeilen beträgt 26.6, der Test liegt bei 25.4. Die einzelne kleine Validation war hier also zu optimistisch (Zufall bei nur 5 Zeilen). Das CV-Mittel passt besser zum Test. Mit 5 Testzeilen ist aber auch der Testfehler selbst nur eine grobe Schätzung.

## B4 · Dasselbe mit `GridSearchCV`

`GridSearchCV` macht B2 und B3 automatisch: alle Kandidaten × alle Folds, danach (`refit=True`) ein Nachtrainieren des besten Setups auf allen übergebenen Daten. Mit demselben `cv` kommt dasselbe heraus.

In [ ]:
search = GridSearchCV(
    base_pipeline,
    param_grid={"model__alpha": alphas},
    cv=cv,
    scoring="neg_mean_absolute_error",
    refit=True,
)
search.fit(X_dev, y_dev)  # 4 candidates x 5 folds = 20 trainings, plus one refit

assert search.best_params_["model__alpha"] == best_alpha_cv
assert abs(-search.best_score_ - cv_errors.loc[best_alpha_cv, "Mittel"]) < 1e-9
grid_test_mae = mean_absolute_error(y_test, search.best_estimator_.predict(X_test))
assert abs(grid_test_mae - test_mae_cv) < 1e-9
print("GridSearchCV liefert dasselbe Ergebnis wie die Schleife in B2 und B3.")

# Teil C: Vorverarbeitungen mit CV vergleichen (und der Leakage-Fehler)

Auch die **Strategie der Imputation** ist eine Entscheidung, die man mit CV treffen kann. Wir vergleichen Median, Mittelwert und KNN-Imputation (fehlender Wert wird aus den 3 ähnlichsten Zeilen berechnet), jeweils mit festem `alpha=1`.

Zusätzlich die **falsche** Variante: Erst auf allen Entwicklungsdaten auffüllen und skalieren, danach die CV. Dort hat jede Validierungszeile den Median und die Skalierung schon mitbestimmt.

In [ ]:
def cv_mae(pipe):
    return -cross_val_score(pipe, X_dev, y_dev, cv=cv, scoring="neg_mean_absolute_error")


candidates = {
    "Median": SimpleImputer(strategy="median"),
    "Mittelwert": SimpleImputer(strategy="mean"),
    "KNN (3 Nachbarn)": KNNImputer(n_neighbors=3),
}

results = {}
for name, imp in candidates.items():
    pipe = Pipeline([("imputer", imp), ("scaler", StandardScaler()), ("model", Ridge(alpha=1))])
    results[f"{name} (richtig, in Pipeline)"] = cv_mae(pipe)

# wrong: preprocessing fitted on all development data before the CV
X_dev_leaky = pd.DataFrame(
    StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(X_dev)),
    index=X_dev.index, columns=X_dev.columns,
)
results["Median (FALSCH: vor der CV auf allen Daten)"] = -cross_val_score(
    Ridge(alpha=1), X_dev_leaky, y_dev, cv=cv, scoring="neg_mean_absolute_error"
)

pd.DataFrame(results, index=[f"Durchlauf {i}" for i in range(1, 6)]).T.assign(
    Mittel=lambda t: t.mean(axis=1)
).round(1)

Lesehilfe: Die drei richtigen Varianten rechnen die Vorverarbeitung in jedem Durchlauf neu. Hier hat KNN das beste Mittel (20.2 gegen 27.5 beim Median), aber die einzelnen Durchläufe schwanken zwischen etwa 13 und 35. Bei nur 20 Zeilen ist ein solcher Vorsprung deshalb mit Vorsicht zu lesen. Die falsche Variante (24.6 statt 27.5 bei gleicher Median-Strategie) sieht besser aus, weil Information aus den Validierungszeilen in Median und Skalierung eingeflossen ist. Dieser Fehler macht die Schätzung optimistisch, ohne dass das Modell wirklich besser ist.

# Teil D: Klassifikation mit `StratifiedKFold`

Wenn eine Klasse selten ist, kann ein normaler `KFold` Folds erzeugen, in denen sie kaum oder gar nicht vorkommt. `StratifiedKFold` hält in jedem Fold den **gleichen Klassenanteil**. Wir machen aus dem Preis eine Klasse: `1` = teuer (die teuersten 30 %), `0` = sonst.

In [ ]:
y_class = (y >= y.quantile(0.70)).astype(int).rename("teuer")
y_class_dev = y_class.loc[X_dev.index]
print("Teure Wohnungen in den Entwicklungsdaten:", int(y_class_dev.sum()), "von", len(y_class_dev))

splitters = {
    "KFold (gemischt)": KFold(n_splits=5, shuffle=True, random_state=SEED),
    "StratifiedKFold": StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
}
rows = []
for name, splitter in splitters.items():
    positives = [int(y_class_dev.iloc[val_pos].sum()) for _, val_pos in splitter.split(X_dev, y_class_dev)]
    rows.append({"Splitter": name, **{f"Fold {i}": f"{p} von 4 teuer" for i, p in enumerate(positives, start=1)}})
pd.DataFrame(rows).set_index("Splitter")

Lesehilfe: Beim normalen `KFold` kann ein Fold zufällig 0 oder viele teure Zeilen enthalten. Beim `StratifiedKFold` ist die Verteilung gleichmässig. Das macht die Scores der Durchläufe vergleichbarer. Jetzt die eigentliche CV mit Pipeline und Klassifikator:

In [ ]:
clf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression()),
])
accuracy = cross_val_score(clf, X_dev, y_class_dev, cv=splitters["StratifiedKFold"], scoring="accuracy")
print("Accuracy pro Durchlauf:", accuracy.round(2))
print(f"Mittel {accuracy.mean():.2f}, Streuung {accuracy.std():.2f}")

clf.fit(X_dev, y_class_dev)
print(f"Test-Accuracy (einmalig, nur 5 Zeilen): {accuracy_score(y_class.loc[X_test.index], clf.predict(X_test)):.2f}")

# Teil E: Zeitreihen mit `TimeSeriesSplit`

Bei Zeitreihen darf das Modell nie auf der **Zukunft** trainieren. Darum gibt es kein Mischen: Jeder Durchlauf trainiert auf einem Stück vom Anfang und validiert auf dem direkt folgenden Stück. Das Trainingsfenster wächst von Durchlauf zu Durchlauf. Die letzten Monate sind das Testset.

In [ ]:
months = pd.RangeIndex(1, 13, name="Monat")
ts_dev = months[:9]  # months 10-12 are the test set
tscv = TimeSeriesSplit(n_splits=4, test_size=1)

ts_plan = pd.DataFrame("-", index=months, columns=[f"Durchlauf {i}" for i in range(1, 5)])
ts_plan.loc[months[9:], :] = "Test"
for i, (train_pos, val_pos) in enumerate(tscv.split(ts_dev), start=1):
    ts_plan.loc[ts_dev[train_pos], f"Durchlauf {i}"] = "Train"
    ts_plan.loc[ts_dev[val_pos], f"Durchlauf {i}"] = "Val"
ts_plan.T

Lesehilfe: In jedem Durchlauf liegt `Val` **nach** allen `Train`-Monaten. Ein Strich (`-`) heisst: In diesem Durchlauf nicht verwendet (liegt noch in der Zukunft). Hier ist nicht mehr jede Zeile einmal `Val`, die ersten Monate sind nie `Val`. Das ist der Preis dafür, dass die Zeit nicht vertauscht wird.

# Zusammenfassung

| | Einfacher Split | Cross-Validation |
|---|---|---|
| Datenaufteilung | Training / Validation / Test = 60 / 20 / 20 | Test = 20 %, Rest in k Folds |
| Modelle in der Auswahl | 1 pro Kandidat | k pro Kandidat (alle neu) |
| Entscheidung | Fehler auf der Validation | Mittelwert und Streuung über k Durchläufe |
| Finales Modell | auf allen Entwicklungsdaten (Training + Validation) neu | genauso: auf allen Entwicklungsdaten neu |
| Bewertung | einmal auf dem Test | einmal auf dem Test |
| Geeignet für | grosse Daten, teures Training | kleine bis mittlere Daten, günstiges Training |